In [1]:
import torch as t
from llama_wrapper import LlamaWrapper
import os
from dotenv import load_dotenv
from matplotlib import pyplot as plt
from IPython.display import display, HTML
import matplotlib
from utils.tokenize import tokenize_llama_chat
from behaviors import get_steering_vector, ALL_BEHAVIORS

/Users/kunalnarwani/Desktop/Delft/Delft/Sem_1/ML_Software_Engineering/raw_code/Learning-Controllable-3H-Representations/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
import os
from llama_wrapper import LlamaWrapper

load_dotenv()
HUGGINGFACE_TOKEN = os.getenv("HF_TOKEN")

model = LlamaWrapper(
    hf_token=HUGGINGFACE_TOKEN,
    model_name="Llama-3.2-1B-Instruct",
)


Loading weights: 100%|██████████| 146/146 [00:00<00:00, 3937.63it/s]


## Step 3 — Mapping middle layers

**Concrete plan:** layer sweep. Generate steering vectors at every layer of Llama-3.2-1B (16 layers), apply each with multiplier ±1 to held-out A/B test questions, and pick the layer with the largest behavioral effect (score(+1) − score(−1)).

In [ ]:
import os
import torch as t
import numpy as np
from tqdm.auto import tqdm

from generate_vectors import generate_save_vectors_for_behavior
from behaviors import (
    REFUSAL,
    get_ab_test_data,
    get_steering_vector,
    get_vector_path,
)
from utils.helpers import get_a_b_probs

BEHAVIOR = REFUSAL
LAYERS = list(range(16))            # Llama-3.2-1B has 16 transformer blocks; use [0,2,4,...] to subsample
MULTIPLIERS = [-1.0, 1.0]
N_TEST = 30                         # cap on held-out items for speed; set to None for full set

# --- Step 3a: generate per-layer vectors (skips work if all already exist) ---
need_gen = [l for l in LAYERS if not os.path.exists(get_vector_path(BEHAVIOR, l, model.model_name_path))]
if need_gen:
    print(f"Generating vectors for {len(need_gen)} layers...")
    generate_save_vectors_for_behavior(
        layers=need_gen, save_activations=False, behavior=BEHAVIOR, model=model,
    )
else:
    print("All layer vectors already exist; skipping generation.")

# --- Step 3b: layer sweep on held-out A/B test set ---
test_data = get_ab_test_data(BEHAVIOR)
if N_TEST is not None:
    test_data = test_data[:N_TEST]

a_id = model.tokenizer.convert_tokens_to_ids("A")
b_id = model.tokenizer.convert_tokens_to_ids("B")
model.set_save_internal_decodings(False)

scores = {m: np.zeros(len(LAYERS)) for m in MULTIPLIERS}  # mean P(matching letter)

for li, layer in enumerate(tqdm(LAYERS, desc="layers")):
    vec = get_steering_vector(BEHAVIOR, layer, model.model_name_path, normalized=False).to(model.device)
    for m in MULTIPLIERS:
        match_probs = []
        for item in test_data:
            model.reset_all()
            model.set_add_activations(layer, m * vec)
            logits = model.get_logits_from_text(user_input=item["question"], model_output="(")
            a_prob, b_prob = get_a_b_probs(logits, a_id, b_id)
            match_probs.append(a_prob if item["answer_matching_behavior"] == "(A)" else b_prob)
        scores[m][li] = float(np.mean(match_probs))

model.reset_all()

effect = scores[1.0] - scores[-1.0]
best_layer = LAYERS[int(np.argmax(effect))]
print(f"Best layer (largest +1 vs -1 gap): {best_layer}  effect={effect.max():.3f}")

# --- plot ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(LAYERS, scores[1.0], marker="o", label="multiplier = +1")
ax.plot(LAYERS, scores[-1.0], marker="o", label="multiplier = -1")
ax.plot(LAYERS, effect, marker="s", linestyle="--", color="black", label="effect = (+1) - (-1)")
ax.axvline(best_layer, color="red", alpha=0.3, linestyle=":", label=f"best layer = {best_layer}")
ax.set_xlabel("layer")
ax.set_ylabel("P(matching-behavior letter)")
ax.set_title(f"Layer sweep on {BEHAVIOR} (n={len(test_data)})")
ax.legend()
plt.show()
